# Module 1 - Segmentation clients

Construire une segmentation RFM simple, puis vérifier comment les segments se répartissent par valeur.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.customer_segmentation import build_rfm, choose_k, fit_segments, profile_segments
INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'transactions_normalized.csv'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'generated'

## 1. Calculer le RFM

La récence est mesurée par rapport à décembre 2020. La fréquence correspond au nombre de lignes de commande et la valeur au total des ventes.

In [ ]:
transactions = pd.read_csv(INPUT_PATH, parse_dates=['Date'])
rfm = build_rfm(transactions, analysis_date='2020-12-01')
rfm.head()

## 2. Choisir le nombre de clusters

On compare les scores de silhouette de 2 à 8 clusters. Le choix final tient compte du score et de l'interprétabilité.

In [ ]:
k_scores = choose_k(rfm)
k_scores

## 3. Construire les segments

Les noms métier sont attribués après le clustering, car les numéros de clusters peuvent changer entre deux exécutions.

In [ ]:
segmented, model, scaler = fit_segments(rfm, n_clusters=4)
profile = profile_segments(segmented)
profile[['Segment', 'n', 'SharePctRevenue']]

## 4. Exporter pour Power BI

La table client et le profil des segments sont les deux sorties principales.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
segmented.to_csv(OUTPUT_DIR / 'customer_segments.csv', index=False)
profile.to_csv(OUTPUT_DIR / 'customer_segment_profiles.csv', index=False)
k_scores.to_csv(OUTPUT_DIR / 'customer_k_selection.csv', index=False)